# Deep Agents 第 1 阶段：harness 基础

对应 `codes/deepagents/index.md` 第 1 阶段，讲义见同目录 `README.md`。

先运行下一格加载 `DEEPSEEK_API_KEY`（可放 `codes/langgraph/local.env`）。每题只改 `# 作答` 下面的代码，做完自己跑通即可，先不要对答案。

In [ ]:
import os
from pathlib import Path

for p in [Path.cwd(), *Path.cwd().parents]:
    env = p / "codes/langgraph/local.env"
    if env.is_file():
        for line in env.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        break

from langchain.chat_models import init_chat_model
model = init_chat_model(f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}")

## 1. 最小 deep agent 与它的工具集

用 `create_deep_agent(model=model)` 建一个代理，让它「把三条 Python 并发要点写进 /notes.md」。

1. 打印 `out.keys()` 和 `out["files"]`。
2. 文件在磁盘上吗？为什么？
3. 工具列表里有没有 `write_todos`？（提示：看 0.7 发布说明）

In [ ]:
from deepagents import create_deep_agent

# 作答


## 2. 打开计划工具

给第 1 题的代理加上 `TodoListMiddleware`，换一个多步任务（例如「对比 asyncio / threading / multiprocessing，分三节写进 /report.md」）。

1. 用 `agent.stream(..., stream_mode="updates")` 观察 `todos` 的状态变化。
2. 和不加时比，模型调用次数和最终文件有什么差别？

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

# 作答


## 3. 子代理与上下文隔离

定义一个 `critic` 子代理（`name` / `description` / `system_prompt`），主代理先写草稿，再派 `critic` 审阅。

1. 在 `out["messages"]` 里找到 `task` 对应的 `ToolMessage`，它包含子代理的哪部分输出？
2. 子代理的 `system_prompt` 出现在父代理消息里了吗？
3. 把 `description` 改成一个含糊的词，再跑一次，主代理还会派活吗？

In [ ]:
# 作答


## 4. 换 backend：长期记忆

用 `CompositeBackend` 把 `/memories/` 路由到 `StoreBackend`（`InMemoryStore`），其余仍走 `StateBackend`。

1. 第一个 `thread_id` 里让代理把「用户偏好：回答简洁」写进 `/memories/pref.md`。
2. 换一个新 `thread_id`，让代理先读这个文件再回答。哪些文件跨线程还在？

注意：用 `checkpointer=MemorySaver()` 才有 thread 概念。

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

# 作答


## 5. 对比题（不写代码）

用自己的话回答：

1. `create_deep_agent` 和第 6 阶段的预置 ReAct agent，在图结构上是不是同一种东西？差别在哪一层？
2. 一个「固定三步」的报表流程，和一个「调研一个陌生领域」的任务，各该选哪个？
3. 为什么说 `write_todos` 是 no-op 工具却依然可能有用？